In [7]:
import mediapipe as mp
import cv2
import numpy as np

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    running_mode=VisionRunningMode.IMAGE
)

with FaceLandmarker.create_from_options(options) as landmarker:
    # 테스트 이미지 로드
    img = cv2.imread('smile.jpg')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    
    result = landmarker.detect(mp_image)
    
    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]
        for b in blendshapes:
            print(f"{b.category_name}: {b.score:.4f}")
    else:
        print("얼굴 감지 안됨")

_neutral: 0.0000
browDownLeft: 0.1296
browDownRight: 0.1051
browInnerUp: 0.0052
browOuterUpLeft: 0.0211
browOuterUpRight: 0.0213
cheekPuff: 0.0006
cheekSquintLeft: 0.0000
cheekSquintRight: 0.0000
eyeBlinkLeft: 0.2252
eyeBlinkRight: 0.2231
eyeLookDownLeft: 0.0385
eyeLookDownRight: 0.0414
eyeLookInLeft: 0.0413
eyeLookInRight: 0.1510
eyeLookOutLeft: 0.1377
eyeLookOutRight: 0.0419
eyeLookUpLeft: 0.2980
eyeLookUpRight: 0.2935
eyeSquintLeft: 0.6295
eyeSquintRight: 0.6461
eyeWideLeft: 0.0038
eyeWideRight: 0.0037
jawForward: 0.0022
jawLeft: 0.0016
jawOpen: 0.0999
jawRight: 0.0002
mouthClose: 0.0027
mouthDimpleLeft: 0.0057
mouthDimpleRight: 0.0126
mouthFrownLeft: 0.0002
mouthFrownRight: 0.0001
mouthFunnel: 0.0466
mouthLeft: 0.0021
mouthLowerDownLeft: 0.1154
mouthLowerDownRight: 0.2212
mouthPressLeft: 0.0652
mouthPressRight: 0.0980
mouthPucker: 0.0112
mouthRight: 0.0005
mouthRollLower: 0.0010
mouthRollUpper: 0.0072
mouthShrugLower: 0.0016
mouthShrugUpper: 0.0179
mouthSmileLeft: 0.9611
mouthSmile

In [66]:
import mediapipe as mp
import cv2
import numpy as np
from PIL import Image

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    running_mode=VisionRunningMode.IMAGE
)

def get_blendshapes(image_path):
    img = Image.open(image_path).convert('RGB')
    img_rgb = np.array(img)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    
    with FaceLandmarker.create_from_options(options) as landmarker:
        result = landmarker.detect(mp_image)
    
    if not result.face_blendshapes:
        return None
    
    return {b.category_name: b.score for b in result.face_blendshapes[0]}

def classify_emotion(bs):
    smile = (bs['mouthSmileLeft'] + bs['mouthSmileRight']) / 2
    eye_squint = (bs['eyeSquintLeft'] + bs['eyeSquintRight']) / 2
    brow_down = (bs['browDownLeft'] + bs['browDownRight']) / 2
    mouth_frown = (bs['mouthFrownLeft'] + bs['mouthFrownRight']) / 2
    eye_wide = (bs['eyeWideLeft'] + bs['eyeWideRight']) / 2
    cheek_squint = (bs['cheekSquintLeft'] + bs['cheekSquintRight']) / 2

    # 진짜 웃음
    if smile > 0.4 and eye_squint > 0.5:
        return 'genuine_smile', +2
    
    # 억지 웃음 (경직)
    elif smile > 0.4 and eye_squint < 0.3:
        return 'forced_smile', 0
    
    # 슬픔/화남
    elif mouth_frown > 0.3 or (brow_down > 0.4 and eye_wide < 0.1):
        return 'sad/angry', -2
    
    # 긴장/경직
    else:
        return 'tense/neutral', 0

def analyze_image(image_path):
    bs = get_blendshapes(image_path)
    if bs is None:
        print("얼굴 감지 안됨")
        return
    
    emotion, score = classify_emotion(bs)
    
    print(f"감정: {emotion}")
    print(f"점수: {score}")
    print(f"--- 주요 수치 ---")
    print(f"mouthSmile: {(bs['mouthSmileLeft']+bs['mouthSmileRight'])/2:.4f}")
    print(f"eyeSquint: {(bs['eyeSquintLeft']+bs['eyeSquintRight'])/2:.4f}")
    print(f"browDown: {(bs['browDownLeft']+bs['browDownRight'])/2:.4f}")
    print(f"mouthFrown: {(bs['mouthFrownLeft']+bs['mouthFrownRight'])/2:.4f}")
    
    return emotion, score

# 테스트
analyze_image('ㅇㅇ.jpg')

감정: tense/neutral
점수: 0
--- 주요 수치 ---
mouthSmile: 0.7896
eyeSquint: 0.4278
browDown: 0.0022
mouthFrown: 0.0000


('tense/neutral', 0)

In [57]:
images = {
    '억지웃음': '억지.jpg',
    '은은한미소': '은은한 미소.jpg',
    '기쁨': 'ㅇㅇ.jpg',
    '활짝웃음': 'smile.jpg'
}

for name, path in images.items():
    bs = get_blendshapes(path)
    smile = (bs['mouthSmileLeft'] + bs['mouthSmileRight']) / 2
    eye_squint = (bs['eyeSquintLeft'] + bs['eyeSquintRight']) / 2
    print(f"{name}: mouthSmile={smile:.4f}, eyeSquint={eye_squint:.4f}")

억지웃음: mouthSmile=0.6435, eyeSquint=0.4493
은은한미소: mouthSmile=0.4972, eyeSquint=0.6526
기쁨: mouthSmile=0.7896, eyeSquint=0.4278
활짝웃음: mouthSmile=0.9580, eyeSquint=0.6378
